<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/16_themes_and_span_coding_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: From Themes to Spans: BERTopic and LLM Span Coding

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

A topic model finds themes nobody defined; a codebook measures themes you did define. This notebook
does both, in that order, on the paragraphs of 60 US inaugural addresses (1789–2025). **BERTopic**
discovers recurring themes and an LLM suggests a name for each; you turn them into a **codebook**.
Then the LLM does **span coding**: for every paragraph, it marks the exact passages that express each
theme.

Spans make the coding **inspectable**. Instead of one label per paragraph, you see exactly which words
the model coded, so you can check them against the text automatically, highlight them, and compare
them word by word with your own coding.

> ⚠️ **Turn on the GPU first:** *Runtime → Change runtime type → T4 GPU.*

In this notebook you will:

- find candidate themes with BERTopic and let an LLM name them
- turn discovered themes into a codebook you control
- code spans, the exact passages for each theme, instead of one label per paragraph
- check spans against the text, highlight them, and measure agreement word by word
- compare spans with topics, and follow themes across two centuries of speeches

**Contents**

1. Discovery, then measurement
2. The data
3. Discovery with BERTopic
4. Naming the themes with an LLM
5. From themes to a codebook
6. Span coding
7. Are the spans in the text?
8. Seeing the spans
9. Agreement, word by word
10. Spans and topics
11. Themes over time
12. Wrap-up

There are 5 exercises along the way, marked ✏️. The solutions notebook has worked answers, but try them yourself first.

## 0. Setup

In [ ]:
!pip install -q bertopic ollama pydantic nltk

import html
import json
import re
import subprocess
import time
from difflib import SequenceMatcher

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import requests
import torch
from bertopic import BERTopic
from hdbscan import HDBSCAN
from IPython.display import HTML, display
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer
from umap import UMAP

SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 140)
print("Running on", device.upper())

---
## 1. Discovery, then measurement

| | Topic model (BERTopic) | Codebook + span coding (LLM) |
|---|---|---|
| Question | which themes are in these texts? | where exactly is theme X, and how much of it? |
| Who defines the themes | the data | you |
| Unit | a whole document, one topic | any passage; one paragraph can have several themes |
| Strength | finds what you didn't expect | measures what you mean, in a way you can check |

Used together, they follow a classic route in qualitative and computational research: explore,
define, then measure and validate (Nelson, 2020). Bonus 11 covers BERTopic in depth; here it is only
the first step.

---
## 2. The data

The inaugural addresses from NLTK, as in Bonus 10. We split each speech into paragraphs, cut very long
paragraphs into chunks of at most 160 words at sentence boundaries, and drop short ones (under 25
words), which are mostly greetings.

In [ ]:
nltk.download("inaugural", quiet=True)
from nltk.corpus import inaugural


def chunks(paragraph, max_words=160):
    sentences = re.split(r"(?<=[.!?])\s+", paragraph)
    out, current = [], []
    for s in sentences:
        if current and len(" ".join(current + [s]).split()) > max_words:
            out.append(" ".join(current))
            current = []
        current.append(s)
    return out + [" ".join(current)] if current else out


rows = []
for f in inaugural.fileids():
    for para in re.split(r"\n\s*\n", inaugural.raw(f)):
        for chunk in chunks(re.sub(r"\s+", " ", para).strip()):
            if len(chunk.split()) >= 25:
                rows.append({"year": int(f[:4]), "president": f[5:-4], "text": chunk})
paras = pd.DataFrame(rows)
paras["pid"] = paras.index
paras["era"] = pd.cut(paras.year, [1788, 1860, 1932, 1980, 2030],
                      labels=["1789-1860", "1861-1932", "1933-1980", "1981-2025"])
print(len(paras), "paragraphs from", paras[["year", "president"]].drop_duplicates().shape[0], "speeches")
paras.groupby("era", observed=True).size()

---
## 3. Discovery with BERTopic

Embed, reduce, cluster, describe, with the settings explained in Bonus 11. Inaugural addresses share a
lot of ceremonial vocabulary, so we add some of it to the stop words.

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)
embeddings = embedder.encode(paras.text.tolist(), show_progress_bar=True, batch_size=64)

STOP = list(ENGLISH_STOP_WORDS | {"shall", "us", "upon", "may", "must", "let", "every", "one", "great",
                                  "today", "people", "nation", "country", "america", "american", "americans"})
topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=SEED),
    hdbscan_model=HDBSCAN(min_cluster_size=20, metric="euclidean", cluster_selection_method="eom",
                          prediction_data=True),
    vectorizer_model=CountVectorizer(stop_words=STOP, ngram_range=(1, 2)),
)
topics, _ = topic_model.fit_transform(paras.text.tolist(), embeddings)
paras["topic"] = topics
info = topic_model.get_topic_info()
print(f"{len(info) - 1} topics; {(paras.topic == -1).mean():.0%} of paragraphs are outliers (-1)")
info.head(12)[["Topic", "Count", "Name"]]

---
## 4. Naming the themes with an LLM

### Starting Ollama

The model runs locally through **Ollama**, as in the optional part of the Day 4 notebook. Colab is
Linux, so we install it here; on your own laptop you'd install the app from
[ollama.com](https://ollama.com) instead. The first cell takes about a minute, the download of the
model a minute or two more.

In [ ]:
!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y pciutils zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):                                       # wait until Ollama answers
    try:
        print(requests.get("http://localhost:11434", timeout=1).text)
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

In [ ]:
MODEL = "qwen2.5:7b"
!ollama pull {MODEL}

In [ ]:
import ollama


def ask(messages, schema=None):
    return ollama.chat(model=MODEL, messages=messages, format=schema,
                       options={"temperature": 0, "num_ctx": 4096})

For each of the ten largest topics, the model sees the top words and three representative paragraphs,
and returns a name and a one-sentence definition. A **schema** (as in Bonus 15) makes sure the
answer has exactly those two fields.

In [ ]:
class ThemeName(BaseModel):
    name: str = Field(description="a short name for the theme, two to five words")
    definition: str = Field(description="one sentence: what a passage must be about to belong to this theme")


def name_topic(t):
    words = ", ".join(w for w, _ in topic_model.get_topic(t)[:10])
    docs = topic_model.get_representative_docs(t)[:3]
    prompt = f"Top words: {words}\n\nRepresentative paragraphs:\n" + "\n\n".join(f"- {d[:600]}" for d in docs)
    reply = ask([{"role": "system", "content": "You help a researcher interpret a topic model of US "
                  "presidential inaugural addresses. Name the theme these paragraphs share, and define it."},
                 {"role": "user", "content": prompt}], schema=ThemeName.model_json_schema())
    return ThemeName.model_validate_json(reply.message.content)


top_topics = info.Topic[info.Topic >= 0].head(10)
names = pd.DataFrame([{"topic": t, "top words": ", ".join(w for w, _ in topic_model.get_topic(t)[:6]),
                       **name_topic(t).model_dump()} for t in top_topics])
names

> **✏️ Exercise 1**
>
> Pick three topics and read five of their paragraphs (`paras[paras.topic == t].text`), not just the top
> words and the name. Does the LLM's name fit what the paragraphs are about? Which topics would you
> merge, and which are too narrow to be a theme?

In [ ]:
# Your code here

---
## 5. From themes to a codebook

The names are suggestions, not results. A codebook needs themes that don't overlap too much, are
broad enough to recur across eras, and come with definitions a coder can apply. Below is a starting
codebook with five themes that run through these speeches. Compare it with your table: merge, rename,
add or drop. It's your measurement instrument.

In [ ]:
CODEBOOK = {
    "Economy and prosperity": "work, wages, trade, taxes, public debt, business, growth, or material well-being",
    "War, peace and security": "war, armed forces, defense, threats from abroad, or peace between nations",
    "Liberty and rights": "freedom, rights, the Constitution, justice, or equality before the law",
    "Unity and national purpose": "the nation as one people, healing divisions, or a shared mission",
    "God and faith": "God, Providence, prayer, blessings, or religious faith",
}

---
## 6. Span coding

The model gets the codebook and one paragraph, and returns a list of **spans**: a theme, and the
passage copied word for word. A paragraph can have several spans, or none. The schema allows only
themes from the codebook; we build it from the codebook, so it changes when you edit the codebook.

In [ ]:
SPAN_INSTRUCTIONS = """You code paragraphs from US presidential inaugural addresses for themes.

Themes:
{codebook}

Find every passage in the paragraph that expresses one of these themes. For each, give the theme and the
passage, copied word for word from the paragraph: a sentence or a clause, never a summary. Code a passage
only if it is clearly about the theme. Most paragraphs have one or two themes; some have none, and then
you return an empty list."""


def span_schema(themes):
    span = {"type": "object", "required": ["theme", "quote"],
            "properties": {"theme": {"type": "string", "enum": list(themes)}, "quote": {"type": "string"}}}
    return {"type": "object", "required": ["spans"], "properties": {"spans": {"type": "array", "items": span}}}


def code_spans(text, codebook=CODEBOOK):
    rules = "\n".join(f"- {name}: {definition}" for name, definition in codebook.items())
    reply = ask([{"role": "system", "content": SPAN_INSTRUCTIONS.format(codebook=rules)},
                 {"role": "user", "content": text}], schema=span_schema(codebook))
    return json.loads(reply.message.content)["spans"]


example = paras[paras.text.str.contains("malice toward none")].iloc[0]
print(example.year, example.president, "\n", example.text, "\n")
code_spans(example.text)

Now a sample: 15 paragraphs from each of four eras, 60 in total. That takes a few minutes.

In [ ]:
coded = paras.groupby("era", observed=True).sample(15, random_state=SEED).reset_index(drop=True)
start = time.time()
coded["spans"] = [code_spans(t) for t in coded.text]
print(f"{len(coded)} paragraphs in {time.time() - start:.0f} s")

spans = pd.DataFrame([{"pid": pid, **s} for pid, ss in zip(coded.pid, coded.spans) for s in ss])
print(f"{len(spans)} spans; paragraphs without any theme: {(coded.spans.str.len() == 0).mean():.0%}")
spans.theme.value_counts()

---
## 7. Are the spans in the text?

We asked for exact words, so we can locate every span in its paragraph: **exact** if it appears word for
word (ignoring case), **partly** if a long stretch of it does (at least 60% of its characters), and
**not in text** otherwise. The positions are what the next sections build on.

In [ ]:
texts = paras.set_index("pid")["text"]


def locate(text, quote):
    """Start and end of the quote in the text, and how well it was found."""
    lower, q = text.lower(), re.sub(r"\s+", " ", quote).strip().strip('"').lower()
    start = lower.find(q) if q else -1
    if start >= 0:
        return start, start + len(q), "exact"
    m = SequenceMatcher(None, lower, q, autojunk=False).find_longest_match(0, len(lower), 0, len(q))
    if q and m.size >= 0.6 * len(q):
        return m.a, m.a + m.size, "partly"
    return np.nan, np.nan, "not in text"


spans[["start", "end", "grounding"]] = pd.DataFrame(
    [locate(texts[p], q) for p, q in zip(spans.pid, spans.quote)], index=spans.index)
spans.grounding.value_counts(normalize=True).round(2)

> **✏️ Exercise 2**
>
> Print the spans that are not `exact`, each with the stretch of the paragraph that `locate` found.
> What did the model do to them? Which would you keep?

In [ ]:
# Your code here

---
## 8. Seeing the spans

With positions, we can highlight the coded passages in the paragraphs. This is the fastest way to
check a coding: you see at a glance what was coded, and what wasn't.

In [ ]:
COLORS = ["#fde68a", "#bfdbfe", "#bbf7d0", "#fecaca", "#e9d5ff", "#fed7aa", "#c7d2fe", "#d9f99d"]


def highlight(text, text_spans, codebook=CODEBOOK):
    color = dict(zip(codebook, COLORS))
    marks = sorted((int(s), int(e), t) for s, e, t in zip(text_spans.start, text_spans.end, text_spans.theme)
                   if not pd.isna(s))
    out, pos = [], 0
    for s, e, t in marks:
        if s < pos:                           # overlapping spans: show the first one
            continue
        out += [html.escape(text[pos:s]),
                f'<mark style="background:{color.get(t, "#e5e7eb")}" title="{html.escape(t)}">'
                f'{html.escape(text[s:e])}</mark>']
        pos = e
    return "".join(out) + html.escape(text[pos:])


def show(rows, text_spans, codebook=CODEBOOK):
    legend = " ".join(f'<mark style="background:{c}">{html.escape(t)}</mark>' for t, c in zip(codebook, COLORS))
    body = "".join(f"<p><b>{r.year} {r.president}</b><br>{highlight(r.text, text_spans[text_spans.pid == r.pid], codebook)}</p>"
                   for r in rows.itertuples())
    display(HTML(f"<div style='max-width:900px; line-height:1.5'>{legend}<hr>{body}</div>"))


show(coded.sample(5, random_state=1), spans)

---
## 9. Agreement, word by word

With spans, agreement isn't "same label or not" but **how many words both codings marked**. For each
theme we compare two codings word by word: the share of coded words that both agree on (an F1 score;
1 is identical, 0 no overlap). Krippendorff's unitizing alpha is the formal version of this idea for
human coders; the F1 is the quick look.

First, a test of **prompt sensitivity**: code 20 paragraphs again with the themes listed in reverse
order. Nothing about the task changed, so in an ideal world the spans would be identical.

In [ ]:
def word_marks(text, text_spans, theme):
    """For every word of the text: is it inside a span of this theme?"""
    inside = np.zeros(len(text), dtype=bool)
    for s, e, t in zip(text_spans.start, text_spans.end, text_spans.theme):
        if t == theme and not pd.isna(s):
            inside[int(s):int(e)] = True
    return np.array([inside[m.start():m.end()].any() for m in re.finditer(r"\S+", text)])


def agreement(spans_a, spans_b, pids, codebook=CODEBOOK):
    rows = []
    for theme in codebook:
        a = np.concatenate([word_marks(texts[p], spans_a[spans_a.pid == p], theme) for p in pids])
        b = np.concatenate([word_marks(texts[p], spans_b[spans_b.pid == p], theme) for p in pids])
        both = (a & b).sum()
        rows.append({"theme": theme, "words A": a.sum(), "words B": b.sum(),
                     "F1": 2 * both / (a.sum() + b.sum()) if a.sum() + b.sum() else np.nan})
    return pd.DataFrame(rows).set_index("theme").round(2)


def to_frame(pids, span_lists):
    df = pd.DataFrame([{"pid": p, **s} for p, ss in zip(pids, span_lists) for s in ss],
                      columns=["pid", "theme", "quote"])
    df[["start", "end", "grounding"]] = pd.DataFrame([locate(texts[p], q) for p, q in zip(df.pid, df.quote)],
                                                     index=df.index, columns=["start", "end", "grounding"])
    return df


reversed_codebook = dict(reversed(list(CODEBOOK.items())))
retest = coded.head(20)
spans_rev = to_frame(retest.pid, [code_spans(t, reversed_codebook) for t in retest.text])
agreement(spans, spans_rev, retest.pid)

Wherever F1 is clearly below 1, the order of the themes in the prompt changed what was coded. That's
worth knowing before you compare themes with each other, and it's a reason to fix the prompt before
you code the full corpus.

> **✏️ Exercise 3**
>
> Code three famous paragraphs yourself: Lincoln 1865 ("malice toward none"), Roosevelt 1933 ("the only
> thing we have to fear") and Kennedy 1961 ("pay any price"). Write your spans as a list of dictionaries
> with `pid`, `theme` and `quote` (copy the words exactly), turn them into a table with `to_frame`, let the
> model code the same paragraphs, and compare with `agreement`. Then look at both codings with `show`.

In [ ]:
# Your code here

---
## 10. Spans and topics

Back to discovery: do the topics BERTopic found line up with the themes we coded? For each topic, the
share of its coded paragraphs that contain each theme.

In [ ]:
present = (spans[spans.grounding != "not in text"].assign(n=1)
           .pivot_table(index="pid", columns="theme", values="n", aggfunc="max", fill_value=0)
           .reindex(coded.pid, fill_value=0))
present["topic"] = coded.set_index("pid")["topic"]
labels = {t: f"{t} · {n}" for t, n in zip(names.topic, names.name)}       # topic number keeps names unique
table = (present[present.topic.isin(labels)].groupby("topic").mean()
         .rename(index=labels).round(2))
table.style.background_gradient(cmap="Greens", axis=None) if len(table) else table

With 60 coded paragraphs spread over many topics, some rows rest on one or two paragraphs, so read this
as a sketch. Where a topic and a theme match, discovery and measurement agree. Where a topic mixes
several themes, the topic model grouped paragraphs by something else: an era, a style, a ceremony.

---
## 11. Themes over time

Spans give a natural measure of emphasis: the **share of words** devoted to each theme. Here by era,
for the 60 coded paragraphs.

In [ ]:
def theme_shares(df_paras, df_spans, codebook=CODEBOOK):
    rows = []
    for r in df_paras.itertuples():
        n_words = len(r.text.split())
        own = df_spans[df_spans.pid == r.pid]
        rows.append({"pid": r.pid, **{t: word_marks(r.text, own, t).sum() / n_words for t in codebook}})
    return pd.DataFrame(rows).set_index("pid")


shares = theme_shares(coded, spans).join(coded.set_index("pid")["era"])
by_era = shares.groupby("era", observed=True).mean()
by_era.plot(marker="o", figsize=(9, 4), ylabel="share of words coded", xlabel="")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
by_era.round(3)

Fifteen paragraphs per era is a small sample, so treat the lines as hypotheses. To measure trends
properly you would code every paragraph (about 1,650, roughly two hours with this model on a T4) or a
much larger sample, after validating the coding on a hand-coded subset.

> **✏️ Exercise 4**
>
> Edit the codebook: add a theme you saw in the topics but not in `CODEBOOK` (for example
> `"Government and its limits"`), or split one that seems too broad. Code 10 paragraphs from `coded` with
> your new codebook (`code_spans(text, my_codebook)`), and compare them with `show`. What changed for the
> other themes?

In [ ]:
# Your code here

> **✏️ Exercise 5**
>
> Compare two speeches in full: code every paragraph of Roosevelt 1933 and Reagan 1981 (both spoke in an
> economic crisis), and compare their theme profiles with `theme_shares`. Do the shares match what you
> know about the two speeches? Highlight a few paragraphs to check.

In [ ]:
# Your code here

---
## 12. Wrap-up

- **Discover, then define, then measure.** A topic model suggests themes; you decide what they
  mean; a validated coding measures them.
- **Names are hypotheses.** Read the paragraphs behind every topic before you trust a label, from a
  person or a model.
- **Spans make coding checkable.** Ask for exact words, locate them automatically, highlight them,
  and compare word by word.
- **Watch the instrument.** The order of the themes and every codebook edit can change the spans:
  fix the prompt and the codebook before the final run, and report them.
- **Validate** against human span coding of a sample (Bonus 5), with a rule for span length in the
  codebook.

In [ ]:
server.terminate()    # stop Ollama when you're done

### Further reading

- Grootendorst, M. (2022). BERTopic: Neural topic modeling with a class-based TF-IDF procedure.
  *arXiv:2203.05794*.
- Nelson, L. K. (2020). Computational grounded theory: A methodological framework. *Sociological
  Methods & Research, 49*(1), 3–42.
- Pham, C. M., Hoyle, A., Sun, S., Resnik, P., & Iyyer, M. (2024). TopicGPT: A prompt-based topic
  modeling framework. *Proceedings of NAACL 2024*.
- Da San Martino, G., Yu, S., Barrón-Cedeño, A., Petrov, R., & Nakov, P. (2019). Fine-grained analysis
  of propaganda in news articles. *Proceedings of EMNLP-IJCNLP 2019*. (Span-level annotation.)
- Krippendorff, K. (2018). *Content analysis: An introduction to its methodology* (4th ed.). Sage.
  (Reliability for unitizing.)